# Big Data Analytics
### Trabajo individual – Convocatoria extraordinaria

---

# Active Learning con Apache Spark

### Diseño y evaluación de una solución escalable para la selección inteligente de instancias basada en incertidumbre y diversidad

---

**Máster Universitario en Ciencia de Datos e Ingeniería de Computadores (Universidad de Granada)**

- **Asignatura:** Big Data II
- **Autor:** David Fernández Martínez  
- **Email:** davidfm8@correo.ugr.es

# Entorno experimental

El proyecto ha sido realizado en un entorno Anaconda con *Python 3.8.10* y *Apache Spark 3.3.0*. La especificación completa del entorno se encuentra disponible en el fichero *enviroment.yml* que está basado en el recomendado en el material de la asignatura procedente del libro *Large-Scale Data Analytics with Python and Spark: A Hands-on Guide to Implementing Machine Learning Solutions: Isaac Triguero and Mikel Galar*.

Los experimentos se han ejecutado en un procesador *Intel Core i5-12500H de 12.ª* generación con $12$ núcleos físicos y $16$ procesadores lógicos, $15.67GB$ de RAM, y un sistema operativo *Microsoft Windows 11 Home*.

### Indice de contenidos
1. [Nested Cross Validation](#nested_cross_validation)
    * [1.1 Limitaciones de los enfoques tradicionales](#limitaciones)
    * [1.2 Estrategia de validación cruzada anidada (Nested Cross Validation)](#explicacion)
        * [1.2.1 Formulación matemática](#formulacion_matematica)
        * [1.2.2 Interpretación del rendimiento](#interpretacion)
    * [1.3 Implementación práctica en este trabajo](#implementacion)    
 
2. [Procedimiento de generación de los folds (reproducibilidad)](#reproducibilidad)
    * [2.1 Formato de los ficheros CSV](#formato_ficheros_csv)
        * [2.1.1 Fichero para Outer CV: *outer_folds.csv*](#outer_folds_csv)
        * [2.1.2 Ficheros para Inner CV: *inner_fold_{outer_fold_idx}.csv*](#inner_folds_csv)    

3. [Validación de los folds creados](#validacion)

# 1. Motivación y contexto

El Active Learning (AL) es un paradigma de aprendizaje automático orientado a escenarios en los que la obtención de etiquetas presenta un coste elevado, debido tanto al tiempo requerido como a la necesidad de intervención de expertos. Por ejemplo, la clasificación de imágenes médicas o la detección de fenómenos físicos pueden requerir conocimientos especializados para llevar a cabo el proceso de etiquetado.

El objetivo de este paradigma es maximizar el rendimiento predictivo del modelo minimizando el número de instancias que deben ser etiquetadas. Para ello, se seleccionan las instancias que pueden aportar mayor información al modelo, por ejemplo, aquellas sobre las que presenta mayor incertidumbre. De este modo, el experto únicamente necesita etiquetar un subconjunto reducido de las muestras disponibles, disminuyendo el coste asociado al proceso de anotación. Las nuevas instancias etiquetadas se incorporan posteriormente al conjunto de entrenamiento, permitiendo reentrenar el modelo con el objetivo de mejorar su capacidad predictiva.

Aunque Active Learning busca reducir el número de instancias que deben ser etiquetadas, la selección de las muestras más informativas requiere evaluar todas las instancias disponibles mediante una estrategia de selección. Cuando el conjunto de datos es masivo, este proceso puede implicar un elevado coste computacional, convirtiendo la propia selección de instancias en un posible cuello de botella. Por tanto, la aplicación de Active Learning sobre grandes volúmenes de datos plantea un problema de procesamiento a gran escala, que requiere soluciones Big Data capaces de distribuir el procesamiento y aprovechar eficientemente los recursos computacionales disponibles.

En este contexto, el objetivo de este trabajo es abordar la siguiente cuestión experimental:

*¿Podemos diseñar una estrategia de selección de instancias para Big Data que evite que su coste computacional se convierta en el cuello de botella de Active Learning?*

# 2. Formulación del problema

Active Learning se plantea como un proceso iterativo de selección y etiquetado de instancias, en el que se parte de un conjunto reducido de muestras etiquetadas (*Labeled, L*) y de un conjunto mayoritario de muestras sin etiquetar (*Unlabeled, U*). En cada iteración, el modelo se entrena utilizando las instancias disponibles en $L$ y se emplea para evaluar las muestras de $U$, con el objetivo de identificar aquellas que presentan un mayor potencial informativo para el aprendizaje. Las instancias seleccionadas son etiquetadas por el oráculo (experto), incorporadas al conjunto $L$ y utilizadas para reentrenar el modelo en la siguiente iteración, tal y como se ilustra en la siguiente figura:

<img src="images/active_learning_schema.png" width="40%">

La estrategia de selección planteada en este trabajo se estructura en dos fases consecutivas:
- **Filtrado basado en incertidumbre**. En esta primera fase se selecciona un subconjunto de instancias de $U$ sobre las que el modelo presenta mayor incertidumbre. De este modo, se descartan aquellas muestras sobre las que el modelo realiza predicciones con elevada confianza y se concentra el proceso de selección en las regiones del espacio de características donde existe una mayor ambigüedad. Además de priorizar instancias potencialmente más informativas, este filtrado reduce el volumen de datos que debe procesarse en la siguiente fase, disminuyendo su coste computacional.
- **Selección basada en diversidad mediante K-Means**. Sobre el conjunto de candidatos obtenido se aplica K-Means para agrupar las instancias en función de la similitud de sus vectores de características. De cada clúster se selecciona la instancia más próxima a su centroide. De este modo, las instancias seleccionadas para etiquetar presentan incertidumbre y diversidad, ya que son potencialmente informativas para el modelo y, al mismo tiempo, cubren diferentes regiones del espacio de características, evitando la redundancia y favoreciendo la representatividad del conjunto de instancias seleccionado.


A continuación, se define formalmente la solución propuesta como un proceso iterativo de $T$ iteraciones, indexadas mediante $t \in \{0, \dots, T-1\}$, a partir de un conjunto de datos definido como:
$$N = \{(x_i, y_i)\}_{i=1}^{|N|}$$

donde $x_i \in \mathbb{R}^d$ representa el vector de características de la instancia $i$ y $y_i$ su etiqueta. Para evaluar la estrategia de selección de instancias, el conjunto $N$ se divide en dos subconjuntos disjuntos, utilizando un $70\%$ de las instancias para entrenamiento y un $30\%$ para evaluación:
$$
\begin{aligned}
|N_{\text{train}}| &= 0.70|N| \\
|N_{\text{test}}| &= 0.30|N|
\end{aligned}
$$

El conjunto de *test* se utiliza únicamente para evaluar el rendimiento predictivo del modelo en cada iteración de *Active Learning*. Por su parte, el conjunto de entrenamiento se divide inicialmente en un conjunto etiquetado $L_0$ y un conjunto no etiquetado $U_0$, de forma que:
$$N_{\text{train}} = L_0 \cup U_0, \quad L_0 \cap U_0 = \emptyset$$

Inicialmente, el $5\%$ del conjunto de entrenamiento se encuentra etiquetado, mientras que el $95\%$ restante permanece sin etiquetar:
$$
\begin{aligned}
|L_0| &= 0.05|N_{\text{train}}| \\
|U_0| &= 0.95|N_{\text{train}}|
\end{aligned}
$$

En cada iteración se selecciona un número $B$ de instancias (*query batch*) para que sean etiquetadas por el experto. Este valor se define como el $1\%$ del conjunto de entrenamiento:
$$B = \lfloor 0.01 |N_{\text{train}}| \rfloor$$

Para determinar el número de instancias consideradas en la fase de filtrado basada en incertidumbre, se utiliza un factor multiplicativo $\alpha$ sobre el tamaño del *query batch*. De este modo, la proporción de instancias de $U_t$, que se pretende conservar en esta fase viene dada por:
$$p_t = \min \left( 1, \frac{\alpha B}{|U_t|} \right)$$

donde $|U_t|$ representa el número de instancias no etiquetadas en la iteración $t$. A partir de esta proporción se define el cuantil:
$$q_t=1-p_t$$

Sea $u(x_i)$ la medida de incertidumbre asociada a una instancia $x_i \in U_t$. El conjunto de candidatos resultante $C_t$ se define seleccionando aquellas instancias cuya incertidumbre es igual o superior al valor correspondiente al cuantil $q_t$ de la distribución de incertidumbre en $U_t$:
$$C_t = \{ x_i \in U_t : u(x_i) \ge Q_{U_t}(q_t) \}$$
donde $Q_{U_t}(q_t)$ representa el cuantil $q_t$ de los valores de incertidumbre de las instancias pertenecientes a $U_t$.

Una vez obtenido el conjunto de candidatos $C_t$, tras el filtrado basado en incertidumbre, se aplica la fase de selección por diversidad mediante *K-Means*. El algoritmo particiona $C_t$ en $B$ clústeres, donde $B$ es el *query batch*. Denotando por $K_j$ el conjunto de instancias asignadas al clúster $j$, para $j \in \{0, \dots, B-1\}$, y por $\mu_j$ su centroide, se selecciona de cada clúster la instancia cuya distancia al centroide es mínima:
$$x_j^* = \arg\min_{x_i \in K_j} d(x_i, \mu_j), \quad j \in \{0, \dots, B-1\}$$

donde $d(⋅,⋅)$ es la función de distancia empleada. De este modo, se obtiene una única instancia representativa de cada clúster y el conjunto de muestras seleccionadas para ser etiquetadas por el oráculo queda definido como:
$$Q_t = \{ x_0^*, x_1^*, \dots, x_{B-1}^* \}$$

Tras el etiquetado de las instancias seleccionadas por parte del oráculo, estas se incorporan al conjunto etiquetado y se eliminan del conjunto no etiquetado:
$$
\begin{aligned}
L_{t+1} &= L_t \cup Q_t \\
U_{t+1} &= U_t \setminus Q_t
\end{aligned}
$$

Finalmente, el modelo se reentrena utilizando el nuevo conjunto etiquetado $L_{t+1}$ y se evalúa sobre el conjunto de test $N_{test}$.
Redactar esto bien:

Una característica fundamental de la formulación propuesta es que los parámetros del algoritmo —el porcentaje inicial de instancias etiquetadas, el tamaño del *query batch* y la proporción de instancias seleccionadas en la fase de filtrado por incertidumbre— se definen de forma proporcional al tamaño del conjunto de datos.

Esta configuración permite evaluar experimentalmente la escalabilidad de la solución al incrementar el tamaño del conjunto de datos, manteniendo las mismas proporciones. Los parámetros de configuración empleados se encuentran definidos en el fichero *scripts/config.py* y se mantienen consistentes en todos los experimentos realizados.

# 3. Diseño de la solución

El código de la solución desarrollada se encuentra en el fichero *scripts/AL_methods.py* estructurado en las siguientes funciones que implementan diferentes aspectos del algoritmo:
- *load_and_preprocess_data()*. Encargado de cargar los datos y realizar el preprocesamiento necesario.
- *train_and_evaluate()*. Encargada de entrenar el modelo sobre los datos etiquetados y evaluar sobre el conjunto de datos de test.
- *get_uncertainty_candidates()*. Implementa la fase de filtrado basado en incertidumbre.
- *diversity_k_means_selection()*. Implementa la fase de selección basada en diversidad mediante *K-Means* y simula el etiquetado del experto de las instancias seleccionadas.

A continuación se explica de forma detallada cada parte de la solución haciendo enfásis en las decisiones tomadas para que la solución sea escalable y eficiente en un entorno distribuido.

## 3.1 *load_and_preprocess_data()*

Esta función realiza las siguientes acciones:
- Carga el conjunto de datos mediante *Spark DataFrame* de forma que *Spark* puede distribuir el procesamiento del dataset entre las particiones disponibles.
- Las características se convierten a *float* y se transforman en un único vector almacenado en una única columna *features* mediante *VectorAssembler*.
- El dataset se divide en train/test mediante la función *randomSplit* de *Spark DataFrame* usando los porcentajes de partición especificados en la variable TRAIN_TEST_SPLIT de *scripts/config.py*.
- Dado que se utiliza *K-Means* en la fase de diversidad se aplica un preprocesamiento de escalado mediante la función *StandardScaler*. Este *scaler* se ajusta únicamente sobre el conjunto de entrenamiento y después se aplica tanto a entrenamiento como a *test*.
- Finalmente, se añade una columna "state" al conjunto de entrenamiento que contiene "L" para las muestras etiquetadas y "U" para las muestras no etiquetadas. La asignación inicial se realiza de forma aletaoria de acuerdo a la variable INITIAL_LABELED_FRACTION de *scripts/config.py*. El DataFrame del conjunto de entrenamiento tiene esta forma:
id_sample (int)  label (int) features (vector) state(string)

La función devuelve el conjunto de entrenamiento y el conjunto de test que son cacheados tras recibirlos de la función. El conjunto de test se mantiene inalterado pero se utiliza en cada iteración de *Active Learning* para evaluar el modelo. En cambio, el conjunto de entrenamiento va cambiando el valor de la columna state en cada iteración, pero en todo el proceso se va utilizando en la cadena de linaje por que todo depende de el por lo que se lee multiples veces. En este caso, después de cada iteración el dataframe de entrenamiento se vuelve a cachear liberando la memoria del dataframe de entrenamiento de la iteración anterior.

## 3.2 *train_and_evaluate()*

Esta función se utiliza para entrenar el modelo sobre los datos etiquetados de la iteración actual y evaluar su calidad mediante el conjunto de *test*. La función está preparada para recibir cualquier estimador de *Spark MLlib* y cualqueir métrica compatible con el evaluador  *MulticlassClassificationEvaluator*. De este modo, el entrenamiento del modelo se está realizando de forma distribuida gracias a la implementación de *Spark MLlib* mientras que la evaluación sobre el conjunto de *test* se realiza distribuida en las particiones del DataFrame.

Esta función devuelve el modelo entrenado para su posterior uso sobre el conjunto no etiquetado ($U$) y el valor de la métrica de evaluación.

En los experimentos realizados en este trabajo se ha utilizado el estimador *LogisticRegression* puesto que se comporta bien con las probabilidades y esto es necesario para el calculo de la incertidumbre. La métrica empleada para evaluar sobre *test* ha sido el *accuracy* dado que se trata de un problema de clasificación binaria balanceado (El conjunto de datos utilizado en los experimentos se presenta en la siguiente sección).


## 3.3 *get_uncertainty_candidates()*

Esta función aplica un filtrado para seleccionar un subconjunto de candidatos del conjunto no etiquetado $U_t$ de la iteración actual de modo que se trabaje sobre las instancias potencialmente más informativas y se reduzca el coste computacional de la siguiente fase de selección por diversidad.

El dataframe de entrenamiento se filtra para seleccionar las muestras no etiquetadas y se utiliza el modelo entrenado, devuelto por la función *train_and_evaluate* para obtener la predicción y probabilidad sobre todo $U_t$. Dado que se trata de un problema de clasificación binaria se ha utilizado la siguiente función de incertidumbre:
$$U(x) = 1 - 2 \left| P(y=1 \mid x) - 0.5 \right|$$
Esta función genera valores entre $0$ y $1$ donde $1$ indica la máxima incertidumbre posible.

Tal y como se ha presentado en la formulación del problema, para decidir el tamaño del subconjunto de candidatos a filtrar en esta fase se define un factor multiplicativo $\alpha$ sobre el *query batch* ($B$). Una opción aparentemente sencilla consistiría en ordenar el dataframe que contiene las muestras no etiquetadas de acuerdo a su valor de incertidumbre y seleccionar las $\alpha \cdot B$ con mayor incertidumbre de la siguiente forma:
```python
candidates_df = unlabeled_df.orderBy("uncertainty", ascending=False).limit(alpha * B)
```

Sin embargo, esta ordenación global puede implicar un proceso de *shuffle* considerable que resulta especialmente costoso cuando el tamaño del conjunto no etiquetado ($U$) es muy elevado. Además, el uso de la acción *limit()* para seleccionar exactamente los $\alpha \cdot B$ con mayor incertidumbre tienden a llevar todos esos datos a una única partición, lo cual empeora el rendimiento del procesamiento posterior o requiere explicitamente una operación de *repartitions()*.

Para evitar que este coste computacional sea un cuello de botella se ha optado por una solución de mayor escalabilidad basada en seleccionar las muestras cuya incertidumbre supera un cuantil que garantiza una proporción de instancias de aproximadamente$\alpha \cdot B$. En esta fase no necesitamos obtener un número exacto de candidatos ni ordenarlos, unicamente necesitamos determinar una frontera aproximada que proporcione un subconjunto adecuado de candidatos más inciertos.

Para ello se utiliza la función *ApproxQuantile* que implementa el algoritmo distribuido *Greenwald-Khanna* que permite obtener el valor de incertidumbre asociado al cuantil que garantiza una proporción de muestras aproximada a  $\alpha \cdot B$.

Usar esta info
>approxQuantile permite calcular de forma distribuida un umbral aproximado sin necesidad de realizar una ordenación global exacta de todas las muestras, reduciendo el coste de esta fase.

Este valor de incertidumbre asociado al cuantil se utiliza para filtrar el dataframe de datos no etiquetados y seleccionar las instancias que superan dicho umbral. Esta operación de filtrado se realiza en cada partición del dataframe de modo que los datos se mantienen distribuidos para las posteriores etapas de procesamiento.

La función devuelve el dataframe con los candidatos filtrados que será utilizado en la siguiente fase.

## 3.4 *diversity_k_means_selection()*

La función *diversity_k_means_selection()* implementa tres procedimientos cuya lógica algorítmica y la distribución de operaciones entre el Driver y los workers se ilustra en la siguiente imagen (puede consultarse en *images/diversity_kmean_selection*). A continuación se explica de forma detallada cada uno de estos procedimientos.
<img src="images/diversity_kmeans_selection.svg" width="80%">

**Agrupamiento de las instancias mediante Bisectic-KMeans**

En primer lugar se aplica el agrupamiento de los candidatos previamente filtrados por incertidumbre mediante un algoritmo de *clustering*. Dado que se desea seleccionar $B$ instancias, el algoritmo se configura con $k=B$ clústeres.

Durante los experimentos realizados se ha observado que el algoritmo *BisecticKMeans* escala mejor que *KMeans* cuando el tamaño de $B$ crece. El algoritmo *K-Means* convencional en cada iteración recalcula las distancias de todos los datos a los k centroides, mientras que en en *BisecticKMeans* se parte de un único clúster y se realizan divisiones sucesivas.  Esta división jerárquica hace que *BisectingKMeans* sea mucho mejor con un $k$ elevado porque, en cada iteración, Spark solo necesita calcular distancias y transmitir por red la información de 2 centroides en lugar de $k$, lo que reduce drásticamente el tráfico entre nodos.

Se opta por Bisecting K-Means frente a K-Means convencional debido a su estrategia de agrupamiento jerárquico mediante divisiones sucesivas, que resulta adecuada para estructurar un conjunto de candidatos potencialmente grande en un número elevado de grupos.


Aunque *K-Means* puede ofrecer un rendimiento más alto en términos de calidad el clustering, en nuestro caso el objetivo es que este procedimiento sea escalable y que ofrezca un agrupamiento con diversidad suficiente para que la selección de muestras sea representativa, no se necesita obtener el mejor clustering posible.

Dado que la implementación de *BisecticKMeans* de *MLlib* es distribuida, tanto el entrenamiento como la asignación de instancias a los clústeres se ejecuta de forma distribuida.

Una vez entrenado el modelo mediante fit(), se aplica transform() sobre el conjunto de candidatos. Esta operación no vuelve a entrenar el modelo, sino que asigna a cada instancia su correspondiente cluster_id, generando clustered_candidates_df, que continúa siendo un DataFrame distribuido y constituye la entrada de la siguiente fase de selección.

**Selección de la instancia más cercana al centroide de cada clúster**

Esta fase recibe clustered_candidates_df, que contiene las instancias candidatas junto con el cluster_id asignado por BisectingKMeans.

El objetivo es obtener una única instancia representativa por clúster, concretamente aquella cuya distancia al centroide sea mínima.

Los centroides se envían mediante broadcast para que todos los *workers* puedan acceder localmente a ellos puesto que todos todos los candidatos pertenecientes a un mismo clúster necesitan consultar el mismo centroide, y los candidatos de un mismo clúster pueden estar distribuidos entre los diferentes *workers*. Sin broadcast, se produciría una transferencia repetida de esta información durante el procesamiento distribuido.

El tamaño de los centroides es pequeño respecto al tamaño de los candidatos. (enlazar mejor con lo de arriba).
El Driver obtiene los centroides del modelo y los distribuye mediante broadcast. Al tratarse de un conjunto reducido (B centroides), cada Worker puede disponer de una copia local sin necesidad de redistribuir el conjunto de candidatos.

El cálculo de la distancia de cada instancia al centroide de su clúster se realiza de forma local en cada *worker*, convirtiendo el dataframe *clustered_candidates_df* a un RDD que contiene tuplas con el formato: 
$$(clave, valor) = (cluster_id, (id_sample, distance))$$
De modo que se conserva unicamente la información necesaria para reducir el tamaño de los datos que se intercambian en el posterior *shuffle*. Solo se conserva la información necesaria para la reducción posterior, minimizando el volumen de datos intercambiado.

Sobre este RDD distribuido se aplica un *reducebyKey()* que realiza una agrupación local en cada *worker* que obtiene el candidato con menor distancia a cada clúster. Los *workers* se comunican mediante *shuffle* intercambiando unicamente el candidato local obtenido y mediante una agregación global por la clave (cluster_id) se obtiene un RDD distribuido con la instancia más cercana a cada clúster, tal y como se ilustra con ejemplos visuales en la imagen.
- *Agregación local*. Se conserva el candidato de menor distancia para cada clúster dentro de cada partición.
- *Shuffle*: Redistribución de los mínimos locales según cluster_id para agrupar los candidatos pertenecientes al mismo clúster.
- *Reducción global:* Se obtiene el candidato de menor distancia de cada clúster.
- Resultado: *closest_per_cluster_rdd* RDD distribuido que contiene un único representante por clúster: (cluster_id, (id_sample, distance)).

Decisión	Justificación
Broadcast de centroides	Los B centroides son pequeños respecto al conjunto de candidatos y permiten calcular las distancias localmente.
Cálculo distribuido mediante map()	Las distancias se calculan sobre las particiones existentes sin centralizar los candidatos en el Driver.
reduceByKey()	Permite realizar agregación local antes del shuffle, reduciendo el volumen de datos intercambiado entre Workers.

**Actualización del estado de etiquetado mediante Broadcast Hash Join**

Esta fase simula el etiquetado del oráculo, modificando la columna "state" del dataframe del conjunto de entrenamiento de las instancias seleccionadas de "U" (unlabeled) a "L" (labeled).

Como ya se ha seleccionado una instancia por clúster, solo se extraen los identificadores de esas instancias del RDD closest_per_cluster_rdd creando un dataframe distribuido (selected_ids_df) con solo esa columna.

Este conjunto de $B$ identificadores se replica localmente en los *workers* mediante *broadctast* lo que permite ejecutar un *join* localmente en cada partición para identificar las muestras seleccionada en el dataframe train_df y actualizar el estado a etiquetado.

Dado que selected_ids_df contiene únicamente $B$ identificadores, mientras que train_df tiene un tamaño considerablemente mayor, el hacer broadcast permite realizar el join local y evitar un *shuffle* del conjunto de entrenamiento lo que reduce el tráfico de red.